# Reset Notebook Configurations

In [ ]:
# Restart the Runtime (Hard Reset)
# This is the most effective way to completely clear RAM and disk cache.
import os
os.kill(os.getpid(), 9)

In [ ]:
# Delete variables
%reset -f

# Clear CUDA cache (only for deep learning examples)
import torch
torch.cuda.empty_cache()

# Clear garbage
import gc
gc.collect()

30

In [ ]:
!rm -rf /content/*
!rm -rf ~/.cache/huggingface

In [ ]:
!df -h       # Disk usage
print("="*100)
print("="*100)
!nvidia-smi  # GPU usage
print("="*100)
print("="*100)
!free -h     # RAM usage

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   65G  43% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
/dev/sda1       119G   72G   48G  61% /opt/bin/.nvidia
tmpfs           6.4G  7.6M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Fri Nov 14 04:26:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

# Code Run

In [1]:
# # Install uv package manager
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.1 MB/s eta 0:00:00


In [2]:
# ============================================================================
# STEP 1: Install Dependencies
# ============================================================================
%%capture
import os

# Install Unsloth and dependencies
if "COLAB_" in "".join(os.environ.keys()):
    !uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !uv pip install --no-deps xformers==0.0.27 trl peft accelerate bitsandbytes triton
else:
    !uv pip install unsloth
    !uv pip install xformers trl peft accelerate bitsandbytes

In [3]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import torch
import pandas as pd
import json

print("✅ All libraries imported successfully!")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ All libraries imported successfully!
Using GPU: Tesla T4


In [4]:
# ============================================================================
# STEP 3: Create Synthetic Company Dataset
# ============================================================================
print("\n📊 Creating synthetic company dataset...")

# Synthetic data about "TechVista Solutions" - a fictional SaaS company
company_data = [
    {
        "question": "What is TechVista Solutions?",
        "answer": "TechVista Solutions is a cloud-based SaaS company founded in 2018, specializing in AI-powered analytics and business intelligence platforms. We serve over 2,500 enterprise clients across 45 countries."
    },
    {
        "question": "Who is the CEO of TechVista Solutions?",
        "answer": "Dr. Sarah Chen has been the CEO of TechVista Solutions since its founding in 2018. She previously worked at Google as a senior ML engineer for 8 years."
    },
    {
        "question": "What products does TechVista offer?",
        "answer": "TechVista offers three main products: DataPulse (real-time analytics), InsightAI (predictive modeling platform), and CloudSync (data integration suite). All products are cloud-native and API-first."
    },
    {
        "question": "Where is TechVista headquartered?",
        "answer": "TechVista Solutions is headquartered in Austin, Texas, with regional offices in London, Singapore, and São Paulo. Our main office is located at 1234 Innovation Drive, Austin, TX 78701."
    },
    {
        "question": "What is TechVista's PTO policy?",
        "answer": "TechVista offers unlimited PTO for all full-time employees. We encourage employees to take at least 3 weeks off per year. Additionally, we provide 12 company holidays and a week-long winter shutdown in December."
    },
    {
        "question": "How do I submit an expense report?",
        "answer": "Expense reports should be submitted through the Concur platform within 30 days of the expense. Include all receipts and categorize expenses correctly. Reports are typically approved within 5-7 business days, and reimbursement occurs in the next pay cycle."
    },
    {
        "question": "What are TechVista's core values?",
        "answer": "TechVista's core values are: Innovation First (embracing new technologies), Customer Obsession (prioritizing client success), Transparency (open communication), Diversity & Inclusion (building diverse teams), and Sustainable Growth (balancing profit with purpose)."
    },
    {
        "question": "What is the company's remote work policy?",
        "answer": "TechVista operates on a hybrid-remote model. Employees can work remotely up to 4 days per week, with at least one day in-office for team collaboration. Fully remote positions are available for certain roles with manager approval."
    },
    {
        "question": "How does TechVista handle data security?",
        "answer": "TechVista is SOC 2 Type II certified and GDPR compliant. We use end-to-end encryption for all data transfers, conduct quarterly security audits, and provide annual security training to all employees. Customer data is stored in geo-specific data centers."
    },
    {
        "question": "What is the promotion process at TechVista?",
        "answer": "TechVista conducts performance reviews twice annually in June and December. Promotions are merit-based and consider project impact, leadership, and skill development. Employees typically need 18-24 months in role before being eligible for promotion."
    },
    {
        "question": "What benefits does TechVista provide?",
        "answer": "TechVista offers comprehensive benefits including health, dental, and vision insurance with 90% premium coverage, 401(k) with 6% company match, annual learning stipend of $2,000, home office stipend of $1,500, and equity grants for all employees."
    },
    {
        "question": "How do I access the company VPN?",
        "answer": "Download the Cisco AnyConnect client from the IT portal at it.techvista.com. Use your company email and SSO credentials to log in. For access issues, contact IT support at support@techvista.com or call ext. 5500."
    },
    {
        "question": "What is TechVista's revenue model?",
        "answer": "TechVista operates on a subscription-based SaaS model with three tiers: Starter ($500/month), Professional ($2,000/month), and Enterprise (custom pricing). We also offer professional services and implementation support at $250/hour."
    },
    {
        "question": "What is the company's hiring process?",
        "answer": "TechVista's hiring process consists of: (1) initial phone screen with recruiting, (2) technical assessment or case study, (3) team interviews (3-4 rounds), (4) final interview with leadership, and (5) reference checks. The entire process typically takes 3-4 weeks."
    },
    {
        "question": "How do I book a conference room?",
        "answer": "Conference rooms can be booked through Google Calendar. All rooms are named after famous scientists (Einstein, Curie, Tesla, etc.). Rooms accommodate 4-16 people and include video conferencing equipment. Book at least 24 hours in advance when possible."
    },
    {
        "question": "What is TechVista's customer support availability?",
        "answer": "Our customer support team is available 24/7 for Enterprise clients. Professional tier clients receive support Monday-Friday, 6 AM - 6 PM local time. Starter tier clients have access to our knowledge base and community forums with email support during business hours."
    },
    {
        "question": "What programming languages does TechVista use?",
        "answer": "TechVista's tech stack includes Python for backend services, TypeScript/React for frontend, Go for microservices, and Rust for performance-critical components. We use PostgreSQL for databases, Redis for caching, and Kafka for event streaming."
    },
    {
        "question": "What is the onboarding process for new employees?",
        "answer": "New employees go through a two-week onboarding program including: Week 1 - orientation, company culture training, IT setup, and department introductions. Week 2 - role-specific training, shadowing team members, and first project assignment. Each new hire is assigned a buddy for their first 90 days."
    },
    {
        "question": "How does TechVista handle performance improvement plans?",
        "answer": "If performance issues arise, managers first provide informal coaching. If issues persist, a formal Performance Improvement Plan (PIP) is initiated for 60-90 days with clear goals and weekly check-ins. HR partners with the manager throughout the process."
    },
    {
        "question": "What is TechVista's environmental sustainability initiative?",
        "answer": "TechVista is committed to carbon neutrality by 2025. We use 100% renewable energy in our data centers, offer EV charging stations at all offices, encourage virtual meetings to reduce travel, and partner with One Tree Planted to offset our carbon footprint."
    },
    {
        "question": "What certifications does TechVista hold?",
        "answer": "TechVista holds SOC 2 Type II, ISO 27001, GDPR compliance, HIPAA compliance for healthcare clients, and PCI DSS for payment processing. We undergo annual audits to maintain these certifications."
    },
    {
        "question": "What is the typical team structure at TechVista?",
        "answer": "Teams follow an agile structure with 5-8 members including a Product Manager, Engineering Manager, 3-5 Software Engineers, a Designer, and a QA Engineer. Teams work in two-week sprints with daily standups and bi-weekly retrospectives."
    },
    {
        "question": "How does TechVista handle customer onboarding?",
        "answer": "New customers receive a dedicated Customer Success Manager for the first 90 days. The onboarding includes: initial kickoff call, data migration assistance, custom training sessions, integration support, and monthly check-ins during the first quarter."
    },
    {
        "question": "What is TechVista's approach to AI and machine learning?",
        "answer": "TechVista uses AI/ML extensively across our products. We employ transformer models for natural language processing, time series forecasting for predictive analytics, and clustering algorithms for customer segmentation. Our ML team consists of 15 data scientists and ML engineers."
    }
]

# Convert to DataFrame and then to Hugging Face Dataset format
df = pd.DataFrame(company_data)
print(f"Created dataset with {len(df)} examples")

# Format for instruction tuning
def format_prompt(row):
    return {
        "text": f"""<|im_start|>system
You are a helpful AI assistant with deep knowledge about TechVista Solutions. Answer questions accurately based on company information.<|im_end|>
<|im_start|>user
{row['question']}<|im_end|>
<|im_start|>assistant
{row['answer']}<|im_end|>"""
    }

formatted_data = [format_prompt(row) for _, row in df.iterrows()]
dataset = Dataset.from_list(formatted_data)

print("✅ Dataset created and formatted!")
print(f"\nSample prompt:\n{formatted_data[0]['text'][:300]}...")


📊 Creating synthetic company dataset...
Created dataset with 24 examples
✅ Dataset created and formatted!

Sample prompt:
<|im_start|>system
You are a helpful AI assistant with deep knowledge about TechVista Solutions. Answer questions accurately based on company information.<|im_end|>
<|im_start|>user
What is TechVista Solutions?<|im_end|>
<|im_start|>assistant
TechVista Solutions is a cloud-based SaaS company founded...


In [5]:
# ============================================================================
# STEP 4: Load Model and Tokenizer
# ============================================================================
print("\n🔧 Loading model and tokenizer...")

max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization for efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Model loaded successfully!")
print(f"Model: Qwen2.5-1.5B-Instruct (4-bit quantized)")


🔧 Loading model and tokenizer...
==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Model loaded successfully!
Model: Qwen2.5-1.5B-Instruct (4-bit quantized)


In [6]:
# ============================================================================
# STEP 5: Configure LoRA for PEFT
# ============================================================================
print("\n⚙️ Configuring LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Optimized for training
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA adapters configured!")


⚙️ Configuring LoRA adapters...


Unsloth 2026.1.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA adapters configured!


In [7]:
# ============================================================================
# STEP 6: Configure Training Arguments
# ============================================================================
print("\n📚 Setting up training configuration...")

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,  # Increase for better results (e.g., 500)
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",  # Change to "wandb" if you want to log
)


📚 Setting up training configuration...


In [8]:
# ============================================================================
# STEP 7: Initialize Trainer
# ============================================================================
print("\n🎯 Initializing trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("✅ Trainer initialized!")



🎯 Initializing trainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/24 [00:00<?, ? examples/s]

✅ Trainer initialized!


In [9]:
# ============================================================================
# STEP 8: Train the Model
# ============================================================================
print("\n🚀 Starting fine-tuning...")
print("This will take a few minutes on a T4 GPU...\n")

trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

The model is already on multiple devices. Skipping the move to device specified in `args`.



🚀 Starting fine-tuning...
This will take a few minutes on a T4 GPU...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 34 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,2.549500
20,1.131100
30,0.656800
40,0.266300
50,0.076400
60,0.045500
70,0.039700
80,0.037700
90,0.036500
100,0.035900



✅ Training complete!
Training loss: 0.4875


In [12]:
# ============================================================================
# STEP 9: Test the Fine-Tuned Model
# ============================================================================
print("\n🧪 Testing the fine-tuned model...\n")

FastLanguageModel.for_inference(model)

test_questions = [

    # Trained questions
    "Who is the CEO of TechVista?",
    "What is the company's remote work policy?",
    "How do I submit an expense report?",

    # Non trained questions
    "Regarding that bad news in public about the finance department fraud, is that true",
    "What is the financial performance for Q4 2025 for this company?"
]

for question in test_questions:
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant with deep knowledge about TechVista Solutions. Answer questions accurately based on company information."},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    answer = response.split("assistant\n")[-1] if "assistant" in response else response

    print(f"Q: {question}")
    print(f"A: {answer}\n")
    print("-" * 80 + "\n")


🧪 Testing the fine-tuned model...

Q: Who is the CEO of TechVista?
A: Dr. Sarah Chen has been the CEO of TechVista Solutions since its founding in 2018. She previously worked at Google as a senior ML engineer for 8 years.

--------------------------------------------------------------------------------

Q: What is the company's remote work policy?
A: TechVista operates on a hybrid-remote model. Employees can work remotely up to 4 days per week, with at least one day in-office for team collaboration. Fully remote positions are available for certain roles with manager approval.

--------------------------------------------------------------------------------

Q: How do I submit an expense report?
A: Expense reports should be submitted through the Concur platform within 30 days of the expense. Include all receipts and categorize expenses correctly. Reports are typically approved within 5-7 business days, and reimbursement occurs in the next pay cycle.

-----------------------------------

Interesting, the model clearly hallucinate here for the 4th and 5th question 🤣🤣.

In [11]:
# ============================================================================
# STEP 10: Save the Model
# ============================================================================
print("\n💾 Saving the model...\n")

# Option 1: Save LoRA adapters only (small file, ~100MB)
model.save_pretrained("techvista_lora_model")
tokenizer.save_pretrained("techvista_lora_model")
print("✅ LoRA adapters saved to 'techvista_lora_model/'")

# Option 2: Save merged model (larger, but standalone)
model.save_pretrained_merged("techvista_merged_model", tokenizer, save_method="merged_16bit")
print("✅ Merged model saved to 'techvista_merged_model/'")


💾 Saving the model...

✅ LoRA adapters saved to 'techvista_lora_model/'


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:41<00:00, 41.14s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:37<00:00, 37.83s/it]


Unsloth: Merge process complete. Saved to `/content/techvista_merged_model`
✅ Merged model saved to 'techvista_merged_model/'


In [ ]:
# # Optional: Push to Hugging Face Hub
# # You'll need to set your HF token first
# """
# from huggingface_hub import login
# login(token="your_hf_token_here")

# model.push_to_hub_merged(
#     "your-username/techvista-qwen-1.5b",
#     tokenizer,
#     save_method="merged_16bit"
# )
# print("✅ Model pushed to Hugging Face Hub!")
# """

# print("\n🎉 Fine-tuning complete! Your model is ready to use.")
# print("\nNext steps:")
# print("1. Test with more questions about TechVista")
# print("2. Increase max_steps to 500+ for better performance")
# print("3. Add more training examples to improve coverage")
# print("4. Export to GGUF format for local inference with Ollama")